In [1]:
# ============================================================
# 04_contagion_baseline
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx

from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/Users/irfanhussain/Desktop/systemic-risk-project"
)

RAW_FFIEC = PROJECT_ROOT / "data" / "raw" / "ffiec"
PROCESSED = PROJECT_ROOT / "data" / "processed"

FFIEC_2025 = RAW_FFIEC / "2025"

print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED)

Project root: /Users/irfanhussain/Desktop/systemic-risk-project
Processed directory: /Users/irfanhussain/Desktop/systemic-risk-project/data/processed


In [2]:
# ============================================================
# Load 300-bank network universe
# ============================================================

top_banks = pd.read_parquet(
    PROCESSED / "mtds_300_bank_universe.parquet"
)

print("Rows:", len(top_banks))
print("Unique banks:", top_banks["IDRSSD"].nunique())

top_banks.head()

Rows: 300
Unique banks: 300


,IDRSSD,FDIC Certificate Number,Reporting Period End Date,Financial Institution Name,Financial Institution State,Financial Institution Filing Type,total_assets,total_equity,loans_net_unearned,total_nonaccrual,...,nonaccrual_to_loan,nonaccrual_to_asset,roa,nii_to_asset,has_nonaccrual,quarterly_interest_expense,quarterly_interest_income,tie_ratio,tier1_capital,t1crc
0,852218,628,2025-06-30,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",OH,031,3788551000,322777000,1387589000,9859000,...,0.007105,0.002602,0.003458,0.006307,1,17391000.0,41286000.0,0.421232,282311000.0,0.074517
1,480228,3510,2025-06-30,"BANK OF AMERICA, NATIONAL ASSOCIATION",NC,031,2665555000,250540000,1137614000,6355000,...,0.005586,0.002384,0.002783,0.005559,1,11375000.0,26192000.0,0.434293,196227000.0,0.073616
2,476810,7213,2025-06-30,"CITIBANK, N.A.",SD,031,1833933000,176115000,687317000,2646000,...,0.003850,0.001443,0.001996,0.006671,1,11393000.0,23627000.0,0.482203,159707000.0,0.087084
3,451965,3511,2025-06-30,"WELLS FARGO BANK, NATIONAL ASSOCIATION",SD,031,1746394000,171723000,889295000,7559000,...,0.008500,0.004328,0.003886,0.007331,1,6408000.0,19211000.0,0.333559,149749000.0,0.085748
4,504713,6548,2025-06-30,U.S. BANK NATIONAL ASSOCIATION,OH,031,671438131,65381988,380242680,1636846,...,0.004305,0.002438,0.002891,0.006167,1,3315992.0,7456695.0,0.444700,60723879.0,0.090439


In [3]:
# ============================================================
# Load MDS representatives
# ============================================================

mds_table = pd.read_parquet(
    PROCESSED / "mtds_representatives.parquet"
)

mds_set = set(
    mds_table["IDRSSD"]
)

print("MDS representatives:", len(mds_set))

MDS representatives: 117


In [4]:
# ============================================================
# Core bank mappings
# ============================================================

banks = (
    top_banks[
        [
            "IDRSSD",
            "Financial Institution Name",
            "total_assets",
            "t1crc"
        ]
    ]
    .copy()
    .sort_values("IDRSSD")
    .reset_index(drop=True)
)

bank_ids = banks["IDRSSD"].tolist()

assets = (
    banks["total_assets"]
    .astype(float)
    .values
)

# Tier-1 capital
tier1_capital = (
    banks["t1crc"].astype(float).values
    * assets
)

asset_map = dict(
    zip(
        bank_ids,
        assets
    )
)

t1_map = dict(
    zip(
        bank_ids,
        tier1_capital
    )
)

name_map = dict(
    zip(
        bank_ids,
        banks["Financial Institution Name"]
    )
)

print("Banks:", len(bank_ids))
print("Total assets:", assets.sum())
print("Total Tier-1 capital:", tier1_capital.sum())

Banks: 300
Total assets: 22407423735.0
Total Tier-1 capital: 1980711590.0


In [7]:
# ============================================================
# Load 2025 distress-risk predictions
# ============================================================

risk_file = (
    PROCESSED /
    "final_systemic_risk_baseline.csv"
)

risk_snapshot = pd.read_csv(risk_file)

# In the saved systemic table, shocked_bank is the bank ID
risk_snapshot["IDRSSD"] = (
    risk_snapshot["shocked_bank"]
    .astype(str)
)

print("Rows:", len(risk_snapshot))
print(
    "Unique banks:",
    risk_snapshot["IDRSSD"].nunique()
)

print(
    "Missing failure probabilities:",
    risk_snapshot["failure_probability"].isna().sum()
)

print(
    "\nFailure probability summary:"
)

print(
    risk_snapshot["failure_probability"].describe()
)

Rows: 300
Unique banks: 300
Missing failure probabilities: 0

Failure probability summary:
count    300.000000
mean       0.020412
std        0.074369
min        0.000018
25%        0.000193
50%        0.000941
75%        0.005498
max        0.772845
Name: failure_probability, dtype: float64


In [8]:
# ============================================================
# Final 2025 risk snapshot for the 300-bank universe
# ============================================================

END_DATE = pd.Timestamp("2025-06-30")

risk_snapshot = risk_snapshot[
    [
        "IDRSSD",
        "failure_probability"
    ]
].copy()

print(
    "Risk scores available for:",
    len(risk_snapshot),
    "banks"
)

Risk scores available for: 300 banks


In [9]:
# ============================================================
# Extract 2025-06-30 aggregate interbank exposures
# ============================================================

RC_PATH = FFIEC_2025 / "part1.txt"

exposure_cols = [
    "IDRSSD",
    "Reporting Period End Date",
    "RCFD0071",
    "RCON0071",
    "RCONB987",
    "RCONB993"
]

rc_exp = pd.read_csv(
    RC_PATH,
    sep="\t",
    dtype=str,
    usecols=exposure_cols
)

# Remove metadata row
if (
    len(rc_exp) > 0
    and pd.isna(rc_exp.iloc[0]["IDRSSD"])
):
    rc_exp = rc_exp.iloc[1:].copy()

# Clean ID
rc_exp["IDRSSD"] = (
    rc_exp["IDRSSD"]
    .astype("string")
    .str.strip()
)

# Parse date
rc_exp["Reporting Period End Date"] = pd.to_datetime(
    rc_exp["Reporting Period End Date"],
    errors="coerce"
)

# Keep only target quarter
rc_exp = rc_exp[
    rc_exp["Reporting Period End Date"]
    == END_DATE
].copy()

# Numeric conversion
for col in [
    "RCFD0071",
    "RCON0071",
    "RCONB987",
    "RCONB993"
]:
    rc_exp[col] = pd.to_numeric(
        rc_exp[col],
        errors="coerce"
    )

# Combine domestic/consolidated reporting fields
rc_exp["interbank_assets"] = (
    rc_exp["RCFD0071"]
    .combine_first(rc_exp["RCON0071"])
)

rc_exp["fed_funds_sold"] = rc_exp["RCONB987"]

rc_exp["fed_funds_purchased"] = rc_exp["RCONB993"]

print("Rows:", len(rc_exp))
print("Unique banks:", rc_exp["IDRSSD"].nunique())
print(
    "Duplicate IDs:",
    rc_exp["IDRSSD"].duplicated().sum()
)

Rows: 4477
Unique banks: 4477
Duplicate IDs: 0


In [10]:
# ============================================================
# 300-bank exposure dataset
# ============================================================

exposure_300 = rc_exp[
    rc_exp["IDRSSD"].isin(bank_ids)
].copy()

print(
    "Matched banks:",
    exposure_300["IDRSSD"].nunique()
)

print(
    "Rows:",
    len(exposure_300)
)

Matched banks: 300
Rows: 300


In [11]:
# ============================================================
# Exposure intensity
# ============================================================

exposure_300["total_assets"] = (
    exposure_300["IDRSSD"]
    .map(asset_map)
)

exposure_300["interbank_asset_ratio"] = (
    exposure_300["interbank_assets"]
    / exposure_300["total_assets"]
)

exposure_300["fed_funds_sold_ratio"] = (
    exposure_300["fed_funds_sold"]
    / exposure_300["total_assets"]
)

exposure_300["fed_funds_purchased_ratio"] = (
    exposure_300["fed_funds_purchased"]
    / exposure_300["total_assets"]
)

print(
    exposure_300[
        [
            "interbank_assets",
            "interbank_asset_ratio",
            "fed_funds_sold_ratio",
            "fed_funds_purchased_ratio"
        ]
    ].describe()
)

       interbank_assets  interbank_asset_ratio  fed_funds_sold_ratio  \
count      3.000000e+02             300.000000            300.000000   
mean       7.544457e+06               0.061956              0.000199   
std        3.510140e+07               0.087696              0.001412   
min        9.591000e+03               0.001203              0.000000   
25%        1.826712e+05               0.017718              0.000000   
50%        4.812585e+05               0.036054              0.000000   
75%        1.348672e+06               0.069225              0.000000   
max        3.970750e+08               0.967413              0.022369   

       fed_funds_purchased_ratio  
count                 300.000000  
mean                    0.002115  
std                     0.009734  
min                     0.000000  
25%                     0.000000  
50%                     0.000000  
75%                     0.000000  
max                     0.092023  


In [12]:
# ============================================================
# Observed aggregate interbank exposure
# ============================================================

ib_map = (
    exposure_300
    .set_index("IDRSSD")["interbank_assets"]
    .astype(float)
    .to_dict()
)

observed_interbank = np.array([
    ib_map[bank]
    for bank in bank_ids
])

print(
    "Total observed interbank assets:",
    observed_interbank.sum()
)

print(
    "Minimum:",
    observed_interbank.min()
)

print(
    "Maximum:",
    observed_interbank.max()
)

Total observed interbank assets: 2263337171.0
Minimum: 9591.0
Maximum: 397075000.0


In [13]:
# ============================================================
# Sparse directed exposure reconstruction
# ============================================================

MIN_COUNTERPARTIES = 5
MAX_COUNTERPARTIES = 20

BASELINE_ALPHA = 0.50
SEEDS = [42, 43, 44, 45, 46]


def build_directed_network(
    seed,
    alpha=BASELINE_ALPHA
):
    """
    Build a sparse directed synthetic exposure matrix.

    W[i, j] = exposure of lender i to borrower j.

    The row sum for every bank exactly preserves
    its observed aggregate interbank assets.
    """

    rng = np.random.default_rng(seed)

    W = np.zeros(
        (len(bank_ids), len(bank_ids)),
        dtype=float
    )

    borrower_weight = (
        assets.astype(float) ** alpha
    )

    borrower_weight /= borrower_weight.sum()

    for i, lender in enumerate(bank_ids):

        target_exposure = ib_map[lender]

        # Number of counterparties
        lender_ratio = (
            assets[i] / assets.max()
        )

        k = int(
            MIN_COUNTERPARTIES
            + lender_ratio
            * (
                MAX_COUNTERPARTIES
                - MIN_COUNTERPARTIES
            )
        )

        k = max(
            MIN_COUNTERPARTIES,
            min(MAX_COUNTERPARTIES, k)
        )

        # Candidate borrowers
        candidates = np.delete(
            np.arange(len(bank_ids)),
            i
        )

        candidate_weights = (
            borrower_weight[candidates]
        )

        candidate_weights /= (
            candidate_weights.sum()
        )

        # Select counterparties
        chosen = rng.choice(
            candidates,
            size=k,
            replace=False,
            p=candidate_weights
        )

        chosen_weights = (
            borrower_weight[chosen]
        )

        chosen_weights /= (
            chosen_weights.sum()
        )

        # Allocate all aggregate exposure
        W[
            i,
            chosen
        ] = (
            target_exposure
            * chosen_weights
        )

    return W

In [14]:
# ============================================================
# Build baseline networks
# ============================================================

baseline_networks = {}

for seed in SEEDS:

    W_seed = build_directed_network(
        seed=seed,
        alpha=BASELINE_ALPHA
    )

    baseline_networks[seed] = W_seed

    print(
        f"Seed {seed}:",
        "non-zero exposures =",
        np.count_nonzero(W_seed)
    )

Seed 42: non-zero exposures = 1552
Seed 43: non-zero exposures = 1552
Seed 44: non-zero exposures = 1552
Seed 45: non-zero exposures = 1552
Seed 46: non-zero exposures = 1552


In [15]:
# ============================================================
# Verify aggregate exposure preservation
# ============================================================

for seed, W_seed in baseline_networks.items():

    reconstructed = W_seed.sum(axis=1)

    absolute_error = np.abs(
        observed_interbank
        - reconstructed
    )

    relative_error = (
        absolute_error
        / observed_interbank
    )

    print(f"\nSeed {seed}")
    print(
        "Max absolute error:",
        absolute_error.max()
    )
    print(
        "Max relative error:",
        relative_error.max()
    )
    print(
        "Observed total:",
        observed_interbank.sum()
    )
    print(
        "Reconstructed total:",
        reconstructed.sum()
    )


Seed 42
Max absolute error: 7.450580596923828e-09
Max relative error: 3.609402490809011e-16
Observed total: 2263337171.0
Reconstructed total: 2263337171.0

Seed 43
Max absolute error: 5.960464477539063e-08
Max relative error: 3.265016942112048e-16
Observed total: 2263337171.0
Reconstructed total: 2263337171.0

Seed 44
Max absolute error: 5.960464477539063e-08
Max relative error: 3.544541373479262e-16
Observed total: 2263337171.0
Reconstructed total: 2263337171.0

Seed 45
Max absolute error: 1.4901161193847656e-08
Max relative error: 3.646207104863349e-16
Observed total: 2263337171.0
Reconstructed total: 2263337171.0

Seed 46
Max absolute error: 5.960464477539063e-08
Max relative error: 3.304455938635326e-16
Observed total: 2263337171.0
Reconstructed total: 2263337171.0


In [16]:
# ============================================================
# Normalize bank IDs across the notebook
# ============================================================

top_banks["IDRSSD"] = (
    top_banks["IDRSSD"]
    .astype("string")
    .str.strip()
)

exposure_300["IDRSSD"] = (
    exposure_300["IDRSSD"]
    .astype("string")
    .str.strip()
)

bank_ids = banks["IDRSSD"].astype("string").tolist()

# Rebuild banks after normalization
banks["IDRSSD"] = (
    banks["IDRSSD"]
    .astype("string")
    .str.strip()
)

bank_ids = banks["IDRSSD"].tolist()

assets = (
    banks["total_assets"]
    .astype(float)
    .values
)

t1_map = dict(
    zip(
        bank_ids,
        (
            banks["t1crc"].astype(float)
            * assets
        )
    )
)

asset_map = dict(
    zip(bank_ids, assets)
)

name_map = dict(
    zip(
        bank_ids,
        banks["Financial Institution Name"]
    )
)

# Rebuild interbank lookup
ib_map = (
    exposure_300
    .set_index("IDRSSD")["interbank_assets"]
    .astype(float)
    .to_dict()
)

observed_interbank = np.array([
    ib_map[b]
    for b in bank_ids
])

print("Banks:", len(bank_ids))
print("Missing interbank exposures:",
      sum(b not in ib_map for b in bank_ids))

Banks: 300
Missing interbank exposures: 0


In [17]:
# ============================================================
# Multi-round contagion simulator
# ============================================================

def simulate_cascade(
    shocked_bank,
    W,
    bank_ids,
    tier1_map,
    lgd=1.0,
    max_rounds=20
):
    """
    W[i, j] = exposure of bank i to bank j.

    A shocked borrower defaults.
    Its lenders suffer LGD * exposure losses.

    A surviving bank fails when cumulative losses
    reach or exceed its Tier-1 capital.
    """

    index = {
        bank_id: i
        for i, bank_id in enumerate(bank_ids)
    }

    shock_idx = index[shocked_bank]
    n = len(bank_ids)

    cumulative_losses = np.zeros(n)

    failed = np.zeros(
        n,
        dtype=bool
    )

    failed[shock_idx] = True

    newly_failed = np.array(
        [shock_idx],
        dtype=int
    )

    rounds = 0

    tier1 = np.array([
        tier1_map[b]
        for b in bank_ids
    ])

    while (
        len(newly_failed) > 0
        and rounds < max_rounds
    ):

        rounds += 1

        round_losses = (
            lgd
            * W[:, newly_failed].sum(axis=1)
        )

        cumulative_losses += round_losses

        candidates = ~failed

        new_failures = (
            candidates
            & (cumulative_losses >= tier1)
        )

        newly_failed = np.where(
            new_failures
        )[0]

        failed[newly_failed] = True

    return {
        "shocked_bank": shocked_bank,
        "total_failed": int(failed.sum()),
        "secondary_failures": int(
            failed.sum() - 1
        ),
        "rounds": int(rounds),
        "total_cumulative_loss": float(
            cumulative_losses.sum()
        ),
        "max_loss_to_tier1": float(
            np.max(
                cumulative_losses / tier1
            )
        )
    }

In [18]:
# ============================================================
# JPMorgan stress test
# ============================================================

largest_bank = (
    banks
    .sort_values(
        "total_assets",
        ascending=False
    )
    .iloc[0]["IDRSSD"]
)

print(
    "Largest bank:",
    name_map[largest_bank]
)

Largest bank: JPMORGAN CHASE BANK, NATIONAL ASSOCIATION


In [19]:
# ============================================================
# JPMorgan across baseline networks
# ============================================================

jpm_results = []

for seed, W_seed in baseline_networks.items():

    result = simulate_cascade(
        shocked_bank=largest_bank,
        W=W_seed,
        bank_ids=bank_ids,
        tier1_map=t1_map,
        lgd=1.0
    )

    jpm_results.append({
        "seed": seed,
        "total_failed": result["total_failed"],
        "secondary_failures": result[
            "secondary_failures"
        ],
        "rounds": result["rounds"],
        "total_cumulative_loss": result[
            "total_cumulative_loss"
        ]
    })

jpm_results_df = pd.DataFrame(
    jpm_results
)

jpm_results_df

,seed,total_failed,secondary_failures,rounds,total_cumulative_loss
0,42,4,3,2,5.629425e+07
1,43,3,2,2,8.868205e+07
2,44,3,2,2,2.800302e+08
3,45,3,2,2,4.274079e+07
4,46,6,5,2,5.334375e+08


In [20]:
# ============================================================
# Baseline systemic impact
# alpha = 0.5
# 5 network reconstructions
# ============================================================

contagion_results = []

for seed, W_seed in baseline_networks.items():

    print(f"Running seed {seed}...")

    for bank in bank_ids:

        result = simulate_cascade(
            shocked_bank=bank,
            W=W_seed,
            bank_ids=bank_ids,
            tier1_map=t1_map,
            lgd=1.0
        )

        contagion_results.append({
            "seed": seed,
            "shocked_bank": bank,
            "total_failed": result[
                "total_failed"
            ],
            "secondary_failures": result[
                "secondary_failures"
            ],
            "rounds": result[
                "rounds"
            ],
            "total_cumulative_loss": result[
                "total_cumulative_loss"
            ],
            "max_loss_to_tier1": result[
                "max_loss_to_tier1"
            ]
        })

contagion_df = pd.DataFrame(
    contagion_results
)

print("\nCompleted.")
print("Rows:", len(contagion_df))

Running seed 42...
Running seed 43...
Running seed 44...
Running seed 45...
Running seed 46...

Completed.
Rows: 1500


In [21]:
# ============================================================
# Average systemic impact across 5 networks
# ============================================================

contagion_summary = (
    contagion_df
    .groupby("shocked_bank")
    .agg(
        mean_secondary_failures=(
            "secondary_failures",
            "mean"
        ),
        std_secondary_failures=(
            "secondary_failures",
            "std"
        ),
        mean_total_failed=(
            "total_failed",
            "mean"
        ),
        mean_cumulative_loss=(
            "total_cumulative_loss",
            "mean"
        ),
        std_cumulative_loss=(
            "total_cumulative_loss",
            "std"
        ),
        max_secondary_failures=(
            "secondary_failures",
            "max"
        )
    )
    .reset_index()
)

contagion_summary["Financial Institution Name"] = (
    contagion_summary["shocked_bank"]
    .map(name_map)
)

contagion_summary = contagion_summary.sort_values(
    [
        "mean_secondary_failures",
        "mean_cumulative_loss"
    ],
    ascending=False
).reset_index(drop=True)

contagion_summary.head(20)

,shocked_bank,mean_secondary_failures,std_secondary_failures,mean_total_failed,mean_cumulative_loss,std_cumulative_loss,max_secondary_failures,Financial Institution Name
0,852218,2.8,1.303840,3.8,2.002370e+08,2.093792e+08,5,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION"
1,451965,2.2,0.836660,3.2,1.620797e+08,8.731820e+07,3,"WELLS FARGO BANK, NATIONAL ASSOCIATION"
2,480228,2.0,0.707107,3.0,1.116308e+08,5.024601e+07,3,"BANK OF AMERICA, NATIONAL ASSOCIATION"
3,476810,1.6,1.140175,2.6,1.817699e+08,9.758551e+07,3,"CITIBANK, N.A."
4,112837,0.8,0.836660,1.8,1.142918e+08,6.318502e+07,2,"CAPITAL ONE, NATIONAL ASSOCIATION"
5,852320,0.8,0.836660,1.8,7.643340e+07,6.031681e+07,2,TRUIST BANK
6,501105,0.6,0.894427,1.6,6.564354e+07,8.338250e+07,2,MANUFACTURERS AND TRADERS TRUST COMPANY
7,817824,0.6,0.547723,1.6,6.373403e+07,6.010840e+07,1,"PNC BANK, NATIONAL ASSOCIATION"
8,75633,0.6,1.341641,1.6,5.649347e+07,6.323534e+07,3,BMO BANK NATIONAL ASSOCIATION
9,504713,0.6,0.547723,1.6,3.839351e+07,2.429499e+07,1,U.S. BANK NATIONAL ASSOCIATION


In [22]:
# ============================================================
# Contagion score
# ============================================================

contagion_summary["cascade_percentile"] = (
    contagion_summary[
        "mean_secondary_failures"
    ].rank(pct=True)
)

contagion_summary["loss_percentile"] = (
    contagion_summary[
        "mean_cumulative_loss"
    ].rank(pct=True)
)

contagion_summary["contagion_score"] = (
    0.5 *
    contagion_summary["cascade_percentile"]
    +
    0.5 *
    contagion_summary["loss_percentile"]
)

contagion_summary = (
    contagion_summary
    .sort_values(
        "contagion_score",
        ascending=False
    )
    .reset_index(drop=True)
)

contagion_summary.head(20)

,shocked_bank,mean_secondary_failures,std_secondary_failures,mean_total_failed,mean_cumulative_loss,std_cumulative_loss,max_secondary_failures,Financial Institution Name,cascade_percentile,loss_percentile,contagion_score
0,852218,2.8,1.303840,3.8,2.002370e+08,2.093792e+08,5,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",1.000000,1.000000,1.000000
1,451965,2.2,0.836660,3.2,1.620797e+08,8.731820e+07,3,"WELLS FARGO BANK, NATIONAL ASSOCIATION",0.996667,0.993333,0.995000
2,476810,1.6,1.140175,2.6,1.817699e+08,9.758551e+07,3,"CITIBANK, N.A.",0.990000,0.996667,0.993333
3,480228,2.0,0.707107,3.0,1.116308e+08,5.024601e+07,3,"BANK OF AMERICA, NATIONAL ASSOCIATION",0.993333,0.986667,0.990000
4,112837,0.8,0.836660,1.8,1.142918e+08,6.318502e+07,2,"CAPITAL ONE, NATIONAL ASSOCIATION",0.985000,0.990000,0.987500
5,852320,0.8,0.836660,1.8,7.643340e+07,6.031681e+07,2,TRUIST BANK,0.985000,0.980000,0.982500
6,501105,0.6,0.894427,1.6,6.564354e+07,8.338250e+07,2,MANUFACTURERS AND TRADERS TRUST COMPANY,0.971667,0.976667,0.974167
7,817824,0.6,0.547723,1.6,6.373403e+07,6.010840e+07,1,"PNC BANK, NATIONAL ASSOCIATION",0.971667,0.970000,0.970833
8,75633,0.6,1.341641,1.6,5.649347e+07,6.323534e+07,3,BMO BANK NATIONAL ASSOCIATION,0.971667,0.966667,0.969167
9,2182786,0.4,0.547723,1.4,9.447307e+07,3.061313e+07,1,GOLDMAN SACHS BANK USA,0.948333,0.983333,0.965833


In [23]:
# ============================================================
# Load 2025 distress probabilities
# ============================================================

risk_file = (
    PROCESSED /
    "final_systemic_risk_baseline.csv"
)

risk_snapshot = pd.read_csv(
    risk_file
)

risk_snapshot["IDRSSD"] = (
    risk_snapshot["shocked_bank"]
    .astype("string")
    .str.strip()
)

risk_snapshot = risk_snapshot[
    [
        "IDRSSD",
        "failure_probability"
    ]
].copy()

# Keep only the 300-bank universe
risk_snapshot = risk_snapshot[
    risk_snapshot["IDRSSD"].isin(bank_ids)
].copy()

print(
    "Risk scores:",
    len(risk_snapshot)
)

print(
    "Missing risk scores:",
    risk_snapshot["failure_probability"].isna().sum()
)

Risk scores: 300
Missing risk scores: 0


In [24]:
# ============================================================
# Final baseline systemic-risk table
# ============================================================

final_systemic = contagion_summary[
    [
        "shocked_bank",
        "Financial Institution Name",
        "mean_secondary_failures",
        "std_secondary_failures",
        "mean_total_failed",
        "mean_cumulative_loss",
        "std_cumulative_loss",
        "contagion_score"
    ]
].copy()

# Add distress probability
risk_map = (
    risk_snapshot
    .set_index("IDRSSD")[
        "failure_probability"
    ]
    .to_dict()
)

final_systemic["failure_probability"] = (
    final_systemic["shocked_bank"]
    .map(risk_map)
)

# Add bank size
final_systemic["total_assets"] = (
    final_systemic["shocked_bank"]
    .map(asset_map)
)

# Distress percentile
final_systemic["distress_percentile"] = (
    final_systemic[
        "failure_probability"
    ].rank(pct=True)
)

# mT-MDS structure
final_systemic["is_mds"] = (
    final_systemic["shocked_bank"]
    .isin(mds_set)
)

print(
    "Missing distress scores:",
    final_systemic["failure_probability"].isna().sum()
)

Missing distress scores: 0


In [25]:
# ============================================================
# 50/50 baseline systemic-risk score
# ============================================================

final_systemic["systemic_risk_score"] = (
    0.5 *
    final_systemic["distress_percentile"]
    +
    0.5 *
    final_systemic["contagion_score"]
)

final_systemic = (
    final_systemic
    .sort_values(
        "systemic_risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

final_systemic[
    [
        "Financial Institution Name",
        "failure_probability",
        "mean_secondary_failures",
        "mean_cumulative_loss",
        "contagion_score",
        "is_mds",
        "systemic_risk_score"
    ]
].head(25)

,Financial Institution Name,failure_probability,mean_secondary_failures,mean_cumulative_loss,contagion_score,is_mds,systemic_risk_score
0,"NORTHERN TRUST COMPANY, THE",0.487829,0.4,3.618409e+07,0.945833,False,0.969583
1,"BANK OF NEW YORK MELLON, THE",0.139585,0.4,5.182261e+07,0.954167,True,0.958750
2,STATE STREET BANK AND TRUST COMPANY,0.123504,0.2,5.323244e+07,0.937500,True,0.947083
3,"MORGAN STANLEY PRIVATE BANK, NATIONAL ASSOCIATION",0.772845,0.2,1.225043e+07,0.889167,True,0.944583
4,UBS BANK USA,0.375422,0.2,1.342468e+07,0.895833,True,0.942917
5,EAST WEST BANK,0.037375,0.4,4.137991e+07,0.952500,False,0.924583
6,BANCO POPULAR DE PUERTO RICO,0.068896,0.2,1.686899e+07,0.902500,False,0.916250
7,"CHARLES SCHWAB BANK, SSB",0.043238,0.2,2.035821e+07,0.907500,True,0.908750
8,"BANK OF AMERICA, NATIONAL ASSOCIATION",0.006026,2.0,1.116308e+08,0.990000,False,0.876667
9,U.S. BANK NATIONAL ASSOCIATION,0.005403,0.6,3.839351e+07,0.960833,True,0.852083


In [26]:
# ============================================================
# Save baseline contagion results
# ============================================================

contagion_summary.to_csv(
    PROCESSED /
    "baseline_contagion_alpha_05.csv",
    index=False
)

contagion_df.to_csv(
    PROCESSED /
    "baseline_contagion_all_shocks_alpha_05.csv",
    index=False
)

final_systemic.to_csv(
    PROCESSED /
    "final_systemic_risk_baseline_v2.csv",
    index=False
)

print("Baseline contagion results saved.")

Baseline contagion results saved.


In [27]:
# ============================================================
# EXPORT ONE BASELINE CONTAGION NETWORK
# ============================================================

# Use the first baseline reconstruction
# (seed 42 was one of the five baseline networks)
W_vis = baseline_networks[42]

contagion_edges = []

n = len(bank_ids)

for i in range(n):
    for j in range(n):
        
        exposure = W_vis[i, j]
        
        if exposure > 0:
            contagion_edges.append({
                "source_id": int(bank_ids[i]),
                "target_id": int(bank_ids[j]),
                "exposure_thousand_usd": float(exposure)
            })

contagion_edges = pd.DataFrame(contagion_edges)

print("Contagion edges:", len(contagion_edges))
print("Total exposure:", contagion_edges["exposure_thousand_usd"].sum())

# ------------------------------------------------------------
# Save edge list
# ------------------------------------------------------------

contagion_edges.to_csv(
    PROCESSED / "contagion_baseline_edges_seed42.csv",
    index=False
)

print(
    "\nSaved:",
    PROCESSED / "contagion_baseline_edges_seed42.csv"
)

Contagion edges: 1552
Total exposure: 2263337171.0

Saved: /Users/irfanhussain/Desktop/systemic-risk-project/data/processed/contagion_baseline_edges_seed42.csv


In [28]:
# ============================================================
# EXPORT CONTAGION NODE INFORMATION
# ============================================================

contagion_nodes = pd.DataFrame({
    "IDRSSD": bank_ids,
    "tier1_capital_thousand_usd": [
        t1_map[b] for b in bank_ids
    ]
})

# Add bank names if name_map exists
if "name_map" in globals():

    contagion_nodes["Financial Institution Name"] = [
        name_map.get(b, str(b))
        for b in bank_ids
    ]

# Add total assets if available
if "assets" in globals():
    contagion_nodes["total_assets_thousand_usd"] = assets

# Add aggregate interbank assets if available
if "interbank_assets" in globals():
    contagion_nodes["interbank_assets_thousand_usd"] = interbank_assets

contagion_nodes.to_csv(
    PROCESSED / "contagion_baseline_nodes.csv",
    index=False
)

print("Nodes exported:", len(contagion_nodes))
print(
    "Saved:",
    PROCESSED / "contagion_baseline_nodes.csv"
)

Nodes exported: 300
Saved: /Users/irfanhussain/Desktop/systemic-risk-project/data/processed/contagion_baseline_nodes.csv
